[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/GCP_Pipeline_Automation.ipynb)

# GCP Pipeline Automation - Cloud Functions, Pub/Sub, Dataproc, Composer

**From manual commands to automated, event-driven production pipeline.**

> Prerequisite: Complete [GCP_Full_Pipeline](GCP_Full_Pipeline.ipynb) first. This notebook automates what you built there.

## What We're Building

*(Diagram: see GitHub for visual version)*
[View on GitHub](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/data/pipelines/cloud/02_Concepts.md)

## Architecture: How All Components Connect

**Incremental Load Flow:**
```
New raw data arrives --> Identify new rows (WHERE load_date > last_run) --> Transform only new rows --> MERGE into silver table
```
[See full diagram on GitHub](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/data/pipelines/cloud/04_Automation.md)

## Each Part of This Notebook

*(Diagram: see GitHub for visual version)*
[View on GitHub](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/data/pipelines/cloud/02_Concepts.md)

## What's Executable vs Explained

| Component | Can Run in Colab? | Cost | This Notebook |
|---|---|---|---|
| **Pub/Sub** | Yes | Free tier | Hands-on |
| **Cloud Functions** | Yes (deploy + trigger) | Free tier | Hands-on |
| **Dataproc (Spark)** | Yes (create, submit, delete) | ~$0.50 for 30 min | Hands-on |
| **Cloud Composer** | No ($300/month) | Too expensive for learning | Code shown, explained |


In [ ]:
# === CONFIGURATION ===
# Same project as GCP_Full_Pipeline
PROJECT_ID = "data-pipeline-project-490820"  # <-- Change to your project
BUCKET_NAME = f"de-accelerator-{PROJECT_ID}"
LOCATION = "us-central1"
REGION = "us-central1"

# Datasets (should already exist from GCP_Full_Pipeline)
RAW_DATASET = "raw"
SILVER_DATASET = "silver"
GOLD_DATASET = "gold"

print(f"Project:  {PROJECT_ID}")
print(f"Bucket:   gs://{BUCKET_NAME}/")
print(f"Region:   {REGION}")

In [ ]:
# === AUTHENTICATE ===
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Colab.")
else:
    print("Running locally. Use 'gcloud auth login' if not authenticated.")

!gcloud config set project {PROJECT_ID}

# Enable required APIs
!gcloud services enable \
    cloudfunctions.googleapis.com \
    pubsub.googleapis.com \
    dataproc.googleapis.com \
    cloudbuild.googleapis.com \
    run.googleapis.com \
    2>&1 | tail -3

print("APIs enabled.")

---

## Prerequisites: Bucket, Data, and BigQuery Datasets

If you're starting fresh (or deleted everything from a previous run), run these cells first. If you already have a bucket and datasets from GCP_Full_Pipeline, skip to the IAM section.


In [ ]:
# === SETUP: Get data files, create bucket, create BigQuery datasets ===
# Skip this cell if you already have these from GCP_Full_Pipeline.

import os

# 1. Get the data files from GitHub
if not os.path.exists("/tmp/de-repo/data/calls.json"):
    print("Downloading data from GitHub...")
    !git clone --depth 1 https://github.com/sunilmogadati/systems-in-production.git /tmp/de-repo 2>/dev/null
    print(f"Data files: {os.listdir('/tmp/de-repo/data/')}")
else:
    print("Data files already exist.")

# 2. Create GCS bucket
!gcloud storage buckets create gs://{BUCKET_NAME}/ --location={LOCATION} 2>&1 || echo "Bucket already exists."

# 3. Upload data to Bronze layer in GCS
!gcloud storage cp /tmp/de-repo/data/calls.json gs://{BUCKET_NAME}/bronze/
!gcloud storage cp /tmp/de-repo/data/campaigns.csv gs://{BUCKET_NAME}/bronze/
!gcloud storage cp /tmp/de-repo/data/orders.csv gs://{BUCKET_NAME}/bronze/
!gcloud storage cp /tmp/de-repo/data/products.csv gs://{BUCKET_NAME}/bronze/
print("\nBronze files uploaded to GCS.")

# 4. Create BigQuery datasets
!bq mk --dataset --location={LOCATION} {PROJECT_ID}:raw 2>&1 || echo "raw dataset exists."
!bq mk --dataset --location={LOCATION} {PROJECT_ID}:silver 2>&1 || echo "silver dataset exists."
!bq mk --dataset --location={LOCATION} {PROJECT_ID}:gold 2>&1 || echo "gold dataset exists."

# 5. Load initial data into BigQuery (so we have a row count to compare)
!bq load --autodetect --replace --source_format=NEWLINE_DELIMITED_JSON raw.calls gs://{BUCKET_NAME}/bronze/calls.json
!bq load --autodetect --replace --source_format=CSV raw.campaigns gs://{BUCKET_NAME}/bronze/campaigns.csv
!bq load --autodetect --replace --source_format=CSV raw.orders gs://{BUCKET_NAME}/bronze/orders.csv
!bq load --autodetect --replace --source_format=CSV raw.products gs://{BUCKET_NAME}/bronze/products.csv

print("\nSetup complete:")
print(f"  Bucket: gs://{BUCKET_NAME}/")
print(f"  Datasets: raw, silver, gold")
print(f"  Bronze tables loaded in BigQuery")


---

## IAM and Permissions Setup

Before any automation works, the right permissions must be in place. Without this, Cloud Functions can't write to BigQuery, Dataproc can't read from GCS, and everything fails silently.

### How GCP Permissions Work

**IAM Permissions:**
```
You (human, Owner role) --> can do everything
Cloud Function (service account) --> needs BigQuery Data Editor + Storage Object Viewer
Dataproc (service account) --> needs Storage Object Admin + BigQuery Data Editor
```
[See full diagram on GitHub](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/data/pipelines/cloud/02_Concepts.md)

**Key concept:** When YOU run commands, you use your own Google account. When a Cloud Function or Dataproc job runs automatically, it uses a **service account** (a robot identity). That service account needs its own permissions.

| Component | Service Account | Roles Needed |
|---|---|---|
| **Cloud Functions** | Default compute SA | BigQuery Data Editor, Storage Object Viewer |
| **Dataproc** | Default compute SA | Storage Object Admin, BigQuery Data Editor |
| **Cloud Composer** | Composer SA | Dataproc Editor, BigQuery Admin, Storage Admin |

### AWS equivalent: IAM Roles + Policies (same concept, different names)


In [ ]:
# === IAM: Grant permissions to ALL service accounts ===
# Cloud Functions (Gen 1) uses the App Engine service account
# Cloud Functions (Gen 2) uses the Compute Engine service account
# We grant permissions to BOTH so either generation works.

import subprocess, json as j

# Get project number (needed for service account emails)
result = subprocess.run(["gcloud", "projects", "describe", PROJECT_ID, "--format=json"],
                        capture_output=True, text=True)
project_info = j.loads(result.stdout)
PROJECT_NUMBER = project_info.get("projectNumber", "UNKNOWN")

# The two service accounts that Cloud Functions use
COMPUTE_SA = f"{PROJECT_NUMBER}-compute@developer.gserviceaccount.com"
APPENGINE_SA = f"{PROJECT_ID}@appspot.gserviceaccount.com"

print(f"Project number: {PROJECT_NUMBER}")
print(f"Compute SA (Gen 2): {COMPUTE_SA}")
print(f"App Engine SA (Gen 1): {APPENGINE_SA}")
print()

# Grant permissions to BOTH service accounts
for SA_EMAIL in [COMPUTE_SA, APPENGINE_SA]:
    print(f"Granting permissions to {SA_EMAIL}...")
    
    # BigQuery Data Editor (write data to tables)
    !gcloud projects add-iam-policy-binding {PROJECT_ID}         --member="serviceAccount:{SA_EMAIL}"         --role="roles/bigquery.dataEditor" --quiet 2>&1 | tail -1
    
    # BigQuery Job User (run load jobs and queries)
    !gcloud projects add-iam-policy-binding {PROJECT_ID}         --member="serviceAccount:{SA_EMAIL}"         --role="roles/bigquery.jobUser" --quiet 2>&1 | tail -1
    
    # Storage Object Admin (read/write GCS)
    !gcloud projects add-iam-policy-binding {PROJECT_ID}         --member="serviceAccount:{SA_EMAIL}"         --role="roles/storage.objectAdmin" --quiet 2>&1 | tail -1
    
    print(f"  Done: {SA_EMAIL}")
    print()

# Dataproc Worker (for Spark section later)
!gcloud projects add-iam-policy-binding {PROJECT_ID}     --member="serviceAccount:{COMPUTE_SA}"     --role="roles/dataproc.worker" --quiet 2>&1 | tail -1

print("All permissions granted:")
print("  - BigQuery Data Editor (write data)")
print("  - BigQuery Job User (run load jobs) <-- required for Cloud Functions to load data")
print("  - Storage Object Admin (read/write GCS)")
print("  - Dataproc Worker (cluster communication)")
print()
print("Granted to both Gen 1 (App Engine) and Gen 2 (Compute) service accounts.")


#### Same Step in GCP Console: IAM Permissions

1. Go to [console.cloud.google.com/iam-admin](https://console.cloud.google.com/iam-admin)
2. Find the service account ending in `@developer.gserviceaccount.com`
3. Click the **pencil icon** (Edit)
4. Click **"Add another role"**
5. Search and add: BigQuery Data Editor, Storage Object Admin, Dataproc Worker
6. Click **"Save"**

**You Should See:** The service account now shows 3+ roles in the IAM table.

**Common error if you skip this:**
```
Access Denied: User does not have bigquery.tables.updateData permission
```
This means the service account doesn't have BigQuery write access.


---

## Part 1: Pub/Sub - The Notification Layer

### What is Pub/Sub?

A message queue. One service PUBLISHES a message. Another service SUBSCRIBES and receives it.

**Analogy:** A bulletin board. Someone pins a note (publish). Anyone watching the board (subscriber) sees it.

**In the pipeline:** GCS publishes "a new file arrived." Your Cloud Function subscribes and gets triggered.

```
GCS bucket          Pub/Sub topic        Cloud Function
(file arrives) ---> (message posted) ---> (code runs)
```

### Why not trigger the function directly from GCS?

You can (and we will). But Pub/Sub adds flexibility:
- Multiple subscribers can react to the same event
- Messages are retained if the subscriber is down (retry built in)
- You can filter messages (only react to certain file types)

### AWS equivalent: SNS (notification) + SQS (queue)

In [ ]:
# === PUB/SUB: Create a topic and subscription ===

TOPIC_NAME = "pipeline-file-events"
SUBSCRIPTION_NAME = "pipeline-file-events-sub"

# Create topic
!gcloud pubsub topics create {TOPIC_NAME} 2>&1 || echo "Topic exists."

# Create subscription
!gcloud pubsub subscriptions create {SUBSCRIPTION_NAME} \
    --topic={TOPIC_NAME} 2>&1 || echo "Subscription exists."

print(f"Topic: {TOPIC_NAME}")
print(f"Subscription: {SUBSCRIPTION_NAME}")

In [ ]:
# === PUB/SUB: Publish a test message ===
# Simulates what GCS would send when a file arrives

import json

test_message = json.dumps({
    "bucket": BUCKET_NAME,
    "name": "bronze/calls/calls_2026_03_15.json",
    "size": "125000",
    "timeCreated": "2026-03-15T02:00:00Z"
})

!gcloud pubsub topics publish {TOPIC_NAME} --message='{test_message}'

print(f"Published message to {TOPIC_NAME}")

In [ ]:
# === PUB/SUB: Receive the message ===
# Pull the message we just published

!gcloud pubsub subscriptions pull {SUBSCRIPTION_NAME} --auto-ack --limit=5

# You Should See: The JSON message we published above
# This is what your Cloud Function would receive when a file lands in GCS

#### Same Step in GCP Console: Pub/Sub

**Create topic:**
1. Go to [console.cloud.google.com/cloudpubsub](https://console.cloud.google.com/cloudpubsub)
2. Click **"Create topic"**
3. **Topic ID:** pipeline-file-events
4. Click **"Create"**

**Create subscription:**
1. Click your topic > **"Create subscription"**
2. **Subscription ID:** pipeline-file-events-sub
3. **Delivery type:** Pull
4. Click **"Create"**

**Publish a test message:**
1. Click your topic > **"Publish message"**
2. Paste JSON in the Message body > Click **"Publish"**

**Pull the message:**
1. Go to your subscription > Click **"Pull"**
2. **You Should See:** Your JSON message in the list


You just saw the full Pub/Sub cycle: create topic, publish message, receive message. In production, GCS publishes automatically and your Cloud Function receives automatically. No manual steps.

---

## Part 2: Cloud Functions - The Trigger Layer

### What is a Cloud Function?

A small piece of code that runs when an event happens. No servers to manage. You deploy it and Google runs it when triggered.

**Analogy:** A doorbell. Press it (event) and the bell rings (code runs). You don't keep someone standing at the door.

**In the pipeline:** When a file lands in GCS, the function loads it into BigQuery.

### AWS equivalent: Lambda

In [ ]:
# === CLOUD FUNCTION: Write the function code ===
# We write the function to a local file, then deploy it

import os
os.makedirs("/tmp/pipeline-function", exist_ok=True)

# main.py - the function code
function_code = '''
from google.cloud import bigquery
import json

def process_new_file(event, context):
    """
    Triggered by a new file in GCS.
    Loads the file into the appropriate BigQuery table.
    """
    bucket = event["bucket"]
    file_name = event["name"]
    
    # Only process files in the bronze/ folder
    if not file_name.startswith("bronze/"):
        print(f"Skipping {file_name} (not in bronze/)")
        return
    
    print(f"Processing: gs://{bucket}/{file_name}")
    
    # Map file path to BigQuery table and format
    if "calls" in file_name:
        table_id = "raw.calls"
        source_format = bigquery.SourceFormat.NEWLINE_DELIMITED_JSON
    elif "orders" in file_name:
        table_id = "raw.orders"
        source_format = bigquery.SourceFormat.CSV
    elif "campaigns" in file_name:
        table_id = "raw.campaigns"
        source_format = bigquery.SourceFormat.CSV
    elif "products" in file_name:
        table_id = "raw.products"
        source_format = bigquery.SourceFormat.CSV
    else:
        print(f"Unknown file type: {file_name}")
        return
    
    # Load into BigQuery (APPEND, not replace)
    client = bigquery.Client()
    job_config = bigquery.LoadJobConfig(
        source_format=source_format,
        autodetect=True,
        write_disposition="WRITE_APPEND",
    )
    
    uri = f"gs://{bucket}/{file_name}"
    load_job = client.load_table_from_uri(uri, table_id, job_config=job_config)
    load_job.result()  # Wait for completion
    
    print(f"Loaded {load_job.output_rows} rows into {table_id}")
    return f"Success: {file_name} -> {table_id}"
'''

# requirements.txt
requirements = 'google-cloud-bigquery>=3.0.0\n'

with open("/tmp/pipeline-function/main.py", "w") as f:
    f.write(function_code)
with open("/tmp/pipeline-function/requirements.txt", "w") as f:
    f.write(requirements)

print("Function code written to /tmp/pipeline-function/")
print("\n--- main.py ---")
print(function_code)

In [ ]:
# === CLOUD FUNCTION: Deploy ===
# This deploys the function to GCP. It triggers on any file created in your bucket.
#
# IMPORTANT: Gen 2 Cloud Functions require these APIs enabled:
#   - Cloud Functions, Cloud Build, Cloud Run, Eventarc, Artifact Registry
# First deploy takes 3-5 minutes (builds a container image).
# If this hangs, check that all APIs are enabled (see cell above).

# Enable ALL required APIs first (this is the most common reason for deployment failure)
!gcloud services enable \
    cloudfunctions.googleapis.com \
    cloudbuild.googleapis.com \
    run.googleapis.com \
    eventarc.googleapis.com \
    artifactregistry.googleapis.com \
    2>&1 | tail -3

print("APIs enabled. Starting deployment (3-5 minutes for first deploy)...")
print("If this hangs for more than 10 minutes, cancel and check the troubleshooting section below.")

!gcloud functions deploy process_new_file \
    --gen2 \
    --runtime=python311 \
    --region={REGION} \
    --source=/tmp/pipeline-function \
    --entry-point=process_new_file \
    --trigger-event-filters="type=google.cloud.storage.object.v1.finalized" \
    --trigger-event-filters="bucket={BUCKET_NAME}" \
    --memory=256MB \
    --timeout=120s \
    --allow-unauthenticated \
    2>&1 | tail -10

# Gen 2 functions require authenticated invocations by default.
# The --allow-unauthenticated flag above handles this.
# If you deployed WITHOUT that flag, run this to fix:
# !gcloud functions add-invoker-policy-binding process_new_file \
#     --region={REGION} --member="allUsers"

print("\nDeployment complete (or check output above for errors).")
print("If deployment failed, try the GCP Console UI method instead (see walkthrough below).")


#### Same Step in GCP Console: Cloud Functions (Gen 2)

**Important:** The GCP Console UI for Gen 2 Cloud Functions uses Eventarc for triggers. The UI can be buggy, especially the bucket browser. Workarounds are noted below.

**Step 1: Enable required APIs** (if not already done)
1. Go to [console.cloud.google.com/apis/library](https://console.cloud.google.com/apis/library)
2. Search and enable each: **Cloud Functions API**, **Cloud Build API**, **Cloud Run API**, **Eventarc API**, **Artifact Registry API**
3. This can take 1-2 minutes per API
4. **Verify:** Go to [console.cloud.google.com/apis/dashboard](https://console.cloud.google.com/apis/dashboard) and confirm all 5 show as enabled

**Step 2: Create the function**
1. Go to [console.cloud.google.com/functions](https://console.cloud.google.com/functions)
2. Click **"Create function"**
3. **Environment:** Select **"2nd gen"**
4. **Function name:** `process_new_file`
5. **Region:** `us-central1`

**Step 3: Configure the trigger**
1. Under **"Trigger"**, click **"Add Eventarc Trigger"** (or it may say "HTTPS" by default — change it)
2. **Trigger type:** If you see a dropdown, select **"Eventarc"**
3. **Event provider:** Select **"Cloud Storage"**
4. **Event:** Select **"google.cloud.storage.object.v1.finalized"**
5. **Bucket:** This is where the UI can be buggy.

   **If the "Browse" button works:**
   - Click **"Browse"** and select your bucket

   **If the "Browse" button doesn't work or hangs:**
   - **Workaround 1:** Type the bucket name directly into the field (without `gs://` prefix). Just the bucket name, e.g., `de-accelerator-my-project-id`
   - **Workaround 2:** Use the CLI instead of the console:
     ```
     gcloud functions deploy process_new_file          --gen2 --runtime=python311 --region=us-central1          --source=/tmp/pipeline-function          --entry-point=process_new_file          --trigger-event-filters="type=google.cloud.storage.object.v1.finalized"          --trigger-event-filters="bucket=YOUR-BUCKET-NAME"          --memory=256MB --timeout=120s
     ```
   - **Workaround 3:** Create a Gen 1 function instead (simpler UI):
     1. Select **"1st gen"** instead of "2nd gen"
     2. **Trigger type:** Cloud Storage
     3. **Event type:** Finalize/Create
     4. **Bucket:** Browse works more reliably in Gen 1
     5. The function code is the same. Gen 1 is slightly older but works fine for this use case.

6. **Service account:** Leave as default (Compute Engine default service account)
7. Click **"Save Trigger"**
8. Click **"Next"**

**Step 4: Add the code**
1. **Runtime:** Python 3.11
2. **Entry point:** `process_new_file`
3. In the **Source code** inline editor:
   - Click on `main.py` in the file list
   - Delete all existing code
   - Paste the function code from the cell above
   - Click on `requirements.txt`
   - Replace contents with: `google-cloud-bigquery>=3.0.0`
4. Click **"Deploy"**

**Step 5: Wait for deployment**
- First deploy takes **3-5 minutes** (it builds a container image)
- You should see a spinner, then a green checkmark
- If it fails, check the error message (most common: missing API enablement)
- **If deploy hangs for more than 10 minutes:** Cancel and try Gen 1 (Workaround 3 above)

**Step 6: Test it**
1. Go to [console.cloud.google.com/storage](https://console.cloud.google.com/storage)
2. Open your pipeline bucket
3. Navigate to the `bronze/` folder (create it if it doesn't exist: click **"Create folder"** > name it `bronze`)
4. Open the `bronze` folder
5. Click **"Upload files"** and upload any JSON or CSV file
6. Wait 30-60 seconds
7. Go back to your Cloud Function > click the function name > **"Logs"** tab
8. **You Should See:** Log entries showing "Processing: gs://..." and "Loaded X rows into raw.calls"

**If nothing appears in logs after 2 minutes:**
- Check the trigger configuration (click function name > **"Trigger"** tab)
- Verify the bucket name matches exactly
- Check the service account has permissions (see IAM section above)

**Troubleshooting:**

| Problem | Cause | Fix |
|---|---|---|
| Browse button for bucket doesn't work | Known Gen 2 UI bug | Type bucket name directly, or use CLI, or use Gen 1 |
| Deploy hangs for 10+ minutes | Cloud Build issue | Cancel, try Gen 1, or use CLI deploy |
| Deploy fails: "API not enabled" | Missing APIs | Enable all 5 APIs in Step 1 |
| "Permission denied" on deploy | Insufficient permissions | Need Editor or Owner role |
| Function deploys but never triggers | Wrong trigger config | Check Trigger tab: must be `object.v1.finalized` on correct bucket |
| Function triggers but fails to load | BigQuery permissions | Grant service account BigQuery Data Editor role |
| "No module named google.cloud" | Missing requirement | Check requirements.txt has `google-cloud-bigquery>=3.0.0` |
| "The request was not authenticated" | Gen 2 requires auth by default | Go to function > Permissions > Grant Access > allUsers > Cloud Run Invoker. Or redeploy with `--allow-unauthenticated` flag |


In [ ]:
# === CLOUD FUNCTION: Test it ===
# Upload a file to GCS. The function should trigger and load it into BigQuery.

# First, check current row count in raw.calls
import subprocess
def run_bq(sql):
    result = subprocess.run(["bq", "query", "--use_legacy_sql=false", "--format=prettyjson", sql],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0: print("ERROR:", result.stderr)

print("Row count BEFORE:")
run_bq(f"SELECT COUNT(*) as row_count FROM `{PROJECT_ID}.raw.calls`")

# Upload the same calls.json again (simulating new data arriving)
!gcloud storage cp /tmp/de-repo/data/calls.json gs://{BUCKET_NAME}/bronze/calls/calls_test_trigger.json

print("\nFile uploaded. Cloud Function should trigger in 10-30 seconds...")
print("Wait 30 seconds, then run the next cell to check.")

In [ ]:
# === VERIFY: Did the Cloud Function load the data? ===
# Run this 30 seconds after uploading the test file

print("Row count AFTER (should be higher):")
run_bq(f"SELECT COUNT(*) as row_count FROM `{PROJECT_ID}.raw.calls`")

# Check function logs
print("\n--- Cloud Function Logs (last 5 entries) ---")
!gcloud functions logs read process_new_file --gen2 --region={REGION} --limit=5 2>&1

#### Same Step in GCP Console: Test and Verify Cloud Function

**Test the trigger:**
1. Go to [console.cloud.google.com/storage](https://console.cloud.google.com/storage)
2. Open your pipeline bucket
3. Navigate to `bronze/` folder (create it if it doesn't exist)
4. Click **"Upload files"**
5. Upload any JSON file (e.g., your calls.json)

**Check the function logs:**
1. Go to [console.cloud.google.com/functions](https://console.cloud.google.com/functions)
2. Click your function name (`process_new_file`)
3. Click the **"Logs"** tab
4. **You Should See:** Within 30-60 seconds, log entries showing:
   - "Processing: gs://your-bucket/bronze/calls.json"
   - "Loaded X rows into raw.calls"
5. If you see errors, check the IAM permissions (function service account needs BigQuery Data Editor)

**Verify in BigQuery:**
1. Go to [console.cloud.google.com/bigquery](https://console.cloud.google.com/bigquery)
2. Run: `SELECT COUNT(*) FROM raw.calls`
3. The count should be higher than before you uploaded


**What just happened:**
1. You uploaded a file to GCS
2. GCS sent a notification to your Cloud Function
3. The function detected the file type (calls = JSON)
4. The function loaded it into BigQuery `raw.calls` (append)
5. All automatic. No commands typed.

This is how Bronze ingestion works in production. Files arrive, the function loads them. 24/7, no human needed.

---

## Part 3: Dataproc (Spark) - The Transform Layer

### Why Spark for Silver?

The BigQuery SQL approach from GCP_Full_Pipeline works fine for our 500-row dataset. At scale (millions of rows, complex transforms, ML features), Spark distributes the work across many machines.

**Same logic (dedup, clean, type-fix). Different execution engine.**

```
BigQuery SQL (what we did):       Spark/Dataproc (what we do now):
  CREATE TABLE silver.calls AS      spark.read.json("gs://bronze/")
  SELECT ... FROM raw.calls         .withColumn(dedup, clean)
  WHERE row_num = 1                 .write.parquet("gs://silver/")
```

### AWS equivalent: EMR (Elastic MapReduce)

In [ ]:
# === DATAPROC: Write the PySpark Silver transform ===

spark_code = '''
# silver_transform.py
# Run on Dataproc: reads Bronze from GCS, writes Silver to GCS as Parquet

import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, row_number, lower, trim, when, lit, hour,
    from_utc_timestamp, date_format
)
from pyspark.sql.window import Window

# Get bucket from command line argument
BUCKET = sys.argv[1] if len(sys.argv) > 1 else "gs://de-accelerator-default"
print(f"Bucket: {BUCKET}")

spark = SparkSession.builder.appName("SilverTransform").getOrCreate()

# =============================================
# READ BRONZE (raw files from GCS)
# =============================================
calls = spark.read.json(f"{BUCKET}/bronze/calls.json")
orders = spark.read.option("header", True).option("inferSchema", True) \\
    .csv(f"{BUCKET}/bronze/orders.csv")

print(f"Bronze calls: {calls.count()} rows")
print(f"Bronze orders: {orders.count()} rows")

# =============================================
# SILVER: Clean calls
# Same logic as BigQuery SQL, but in PySpark
# =============================================

# Dedup by call_id (keep first by start_time)
window = Window.partitionBy("call_id").orderBy("start_time")

silver_calls = (calls
    .withColumn("row_num", row_number().over(window))
    .filter(col("row_num") == 1)
    .drop("row_num")

    # Timezone: UTC to Eastern
    .withColumn("call_started_local",
                from_utc_timestamp(col("start_time"), "US/Eastern"))
    .withColumn("call_ended_local",
                from_utc_timestamp(col("end_time"), "US/Eastern"))
    .withColumn("call_date_local",
                date_format(col("call_started_local"), "yyyy-MM-dd"))
    .withColumn("call_hour_local",
                hour(col("call_started_local")))

    # Standardize text
    .withColumn("disposition", lower(trim(col("disposition"))))
    .withColumn("call_type", trim(col("channel")))

    # Flag nulls (don't drop)
    .withColumn("has_missing_duration", col("duration").isNull())
    .withColumn("has_missing_disposition", col("disposition").isNull())
)

# SILVER: Clean orders
window_orders = Window.partitionBy("order_id").orderBy(col("order_date").desc())
silver_orders = (orders
    .withColumn("row_num", row_number().over(window_orders))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

# =============================================
# WRITE SILVER to GCS as Parquet
# =============================================
silver_calls.write.mode("overwrite") \\
    .partitionBy("call_date_local") \\
    .parquet(f"{BUCKET}/silver/calls/")

silver_orders.write.mode("overwrite") \\
    .parquet(f"{BUCKET}/silver/orders/")

print(f"Silver calls: {silver_calls.count()} rows")
print(f"Silver orders: {silver_orders.count()} rows")
print("Silver transform complete.")

spark.stop()
'''

with open("/tmp/silver_transform.py", "w") as f:
    f.write(spark_code)

# Upload to GCS
!gcloud storage cp /tmp/silver_transform.py gs://{BUCKET_NAME}/code/

print("PySpark Silver transform uploaded to GCS.")
print("\n--- silver_transform.py ---")
print(spark_code[:500] + "\n...")

In [ ]:
# === DATAPROC: Create a cluster ===
# This takes 2-3 minutes. Cost: ~$0.15 for 30 minutes with this config.

CLUSTER_NAME = "pipeline-silver-cluster"

!gcloud dataproc clusters create {CLUSTER_NAME} \
    --region={REGION} \
    --num-workers=2 \
    --worker-machine-type=n1-standard-2 \
    --worker-boot-disk-size=100 \
    --master-boot-disk-size=100 \
    --master-machine-type=n1-standard-2 \
    --image-version=2.1-debian11 \
    --max-idle=600s

# If you get "does not have enough resources" error, specify a different zone:
# Add: --zone=us-central1-b  (or us-central1-c, us-central1-f)
# If ALL zones fail, try a different region: --region=us-east1 \
    2>&1 | tail -5

# --max-idle=600s

# If you get "does not have enough resources" error, specify a different zone:
# Add: --zone=us-central1-b  (or us-central1-c, us-central1-f)
# If ALL zones fail, try a different region: --region=us-east1: auto-delete after 10 minutes of no jobs (cost protection)

print(f"\nCluster '{CLUSTER_NAME}' created.")

#### Same Step in GCP Console: Dataproc

**Create cluster:**
1. Go to [console.cloud.google.com/dataproc](https://console.cloud.google.com/dataproc)
2. Click **"Create cluster"**
3. **Name:** pipeline-silver-cluster | **Region:** us-central1
4. **Master:** n1-standard-2 (1 node) | **Workers:** n1-standard-2 (2 nodes)
5. Under **"Manage cluster"**: set **Idle delete** to 10 minutes
6. Click **"Create"** (takes 2-3 minutes)

**Submit a job:**
1. Click cluster name > **"Submit job"**
2. **Job type:** PySpark
3. **Main file:** gs://your-bucket/code/silver_transform.py
4. **Arguments:** gs://your-bucket
5. Click **"Submit"**

**You Should See:** Job status "Succeeded" with output logs.

**Delete cluster:** Check box next to cluster > **"Delete"**. Always delete when done (~$0.50/hour).


**Troubleshooting:**

| Problem | Cause | Fix |
|---|---|---|
| "does not have enough resources" | Zone is full (capacity issue) | Add `--zone=us-central1-b` (or try `-c`, `-f`). In Console: select a specific zone instead of "Auto" |
| "does not have enough resources" in ALL zones | Region capacity issue | Use `--region=us-east1` instead. In Console: change region |
| "Failed to validate permissions for service account" | Cloud Resource Manager API not enabled | Run: `gcloud services enable cloudresourcemanager.googleapis.com` |
| "DISKS_TOTAL_GB quota exceeded" | Default disk size too large | Use `--worker-boot-disk-size=100 --master-boot-disk-size=100` |
| Cluster stuck in PROVISIONING | Taking longer than usual | Wait up to 5 minutes. If still stuck, delete and recreate in a different zone |


In [ ]:
# === DATAPROC: Submit the Silver transform job ===

!gcloud dataproc jobs submit pyspark \
    gs://{BUCKET_NAME}/code/silver_transform.py \
    --cluster={CLUSTER_NAME} \
    --region={REGION} \
    -- gs://{BUCKET_NAME} \
    2>&1 | tail -20

print("\nSpark job complete.")

In [ ]:
# === VERIFY: Check Silver output in GCS ===

print("Silver calls (Parquet files in GCS):")
!gcloud storage ls gs://{BUCKET_NAME}/silver/calls/ 2>&1 | head -10

print("\nSilver orders (Parquet files in GCS):")
!gcloud storage ls gs://{BUCKET_NAME}/silver/orders/ 2>&1 | head -5

# Load Silver Parquet into BigQuery for Gold layer
print("\nLoading Silver Parquet into BigQuery...")
!bq load --source_format=PARQUET --replace \
    {SILVER_DATASET}.calls \
    'gs://{BUCKET_NAME}/silver/calls/*.parquet' 2>&1 | tail -3

!bq load --source_format=PARQUET --replace \
    {SILVER_DATASET}.orders \
    'gs://{BUCKET_NAME}/silver/orders/*.parquet' 2>&1 | tail -3

print("\nSilver data loaded into BigQuery. Ready for Gold.")

#### Same Step in GCP Console: Verify Spark Output

**Check Silver files in GCS:**
1. Go to [console.cloud.google.com/storage](https://console.cloud.google.com/storage)
2. Open your pipeline bucket
3. Navigate to `silver/calls/` folder
4. **You Should See:** Parquet files (`.parquet` extension), possibly partitioned by date
5. Navigate to `silver/orders/` — same thing

**Check Dataproc job status:**
1. Go to [console.cloud.google.com/dataproc/jobs](https://console.cloud.google.com/dataproc/jobs)
2. Find your job (most recent)
3. Click it to see output logs
4. **You Should See:** "Silver calls: X rows" and "Silver transform complete."

**Load Silver into BigQuery (if not done by CLI):**
1. Go to BigQuery Console
2. Click `silver` dataset > **"Create table"**
3. **Source:** Google Cloud Storage
4. **Path:** Browse to `silver/calls/*.parquet`
5. **File format:** Parquet
6. **Table name:** `calls`
7. Click **"Create table"**
8. Repeat for `silver/orders/`


#### Before You Delete: Explore the Spark UI

The Spark UI shows how your job actually executed: the DAG, stages, tasks, shuffle, memory usage. It is only available while the cluster is running. Once you delete the cluster, the UI is gone.

**Via GCP Console:**
1. Go to [console.cloud.google.com/dataproc/clusters](https://console.cloud.google.com/dataproc/clusters)
2. Click your cluster name (`pipeline-silver-cluster`)
3. Click the **"Web Interfaces"** tab
4. Click **"Spark History Server"**

**What to look at:**

| Tab | What It Shows | What to Look For |
|---|---|---|
| **Jobs** | List of completed Spark jobs | Your Silver transform job |
| **Stages** | Each job broken into stages | How many stages (each shuffle = new stage) |
| **DAG Visualization** | Click a stage to see the execution graph | Which operations ran in sequence vs parallel |
| **SQL** | Query plans for DataFrame operations | The logical and physical plan Spark chose |
| **Executors** | Worker nodes and resource usage | Memory per executor, task distribution, GC time |

**How to see the DAG:**
1. In Spark History Server, click your job
2. Click a Stage number
3. Click **"DAG Visualization"**
4. You'll see boxes connected by arrows: each box is an operation (read, filter, join, write). Arrows show data flow. Color coding shows narrow (green) vs wide (blue/red) transforms.

**What to check before deleting the cluster:**
- Did the job succeed? (green status)
- How long did each stage take? (spot bottlenecks)
- Was there a large shuffle? (indicates expensive join or groupBy)
- Were executors evenly loaded? (skew = one executor doing most work)

**Via CLI (get the web interface URL):**
```bash
gcloud dataproc clusters describe pipeline-silver-cluster \
    --region=us-central1 \
    --format="value(config.endpointConfig.httpPorts)"
```

**After you've explored the Spark UI, proceed to delete the cluster below.**


In [ ]:
# === DATAPROC: Delete the cluster (stop paying) ===
# IMPORTANT: Always delete when done.

!gcloud dataproc clusters delete {CLUSTER_NAME} \
    --region={REGION} \
    --quiet \
    2>&1 | tail -3

print(f"Cluster '{CLUSTER_NAME}' deleted. No more charges.")

---

## Part 3b: Silver to Gold (Completing the Pipeline)

After Spark writes Silver Parquet to GCS and we load it into BigQuery, we need to build the Gold layer. This is the same SQL from GCP_Full_Pipeline, now part of the automated flow.

**In production, Cloud Composer runs these steps automatically after Dataproc finishes.**
In this notebook, we run them manually to see each step.

```
Spark wrote Silver Parquet to GCS
    |
    v  bq load (done in verify step above)
BigQuery silver.calls, silver.orders
    |
    v  SQL: CREATE TABLE gold.dim_* and gold.fact_calls
BigQuery gold.* (star schema, business-ready)
    |
    v  SQL quality gates
Validated and ready for reports
```


In [ ]:
# === GOLD: Build dimensions and fact table ===
# Same SQL as GCP_Full_Pipeline, completing the automated pipeline

R = f"{PROJECT_ID}.{RAW_DATASET}"
S = f"{PROJECT_ID}.{SILVER_DATASET}"
G = f"{PROJECT_ID}.{GOLD_DATASET}"

# Create Gold dataset
!bq mk --dataset --location={LOCATION} {PROJECT_ID}:{GOLD_DATASET} 2>&1 || echo "Dataset exists."

import subprocess
def run_bq(sql):
    result = subprocess.run(["bq", "query", "--use_legacy_sql=false", "--format=prettyjson", sql],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0: print("ERROR:", result.stderr)

# dim_date
run_bq(f"""
CREATE OR REPLACE TABLE `{G}.dim_date` AS
SELECT
    CAST(FORMAT_DATE('%Y%m%d', d) AS INT64) AS date_key,
    d AS full_date,
    FORMAT_DATE('%A', d) AS day_name,
    EXTRACT(DAYOFWEEK FROM d) IN (1, 7) AS is_weekend,
    EXTRACT(MONTH FROM d) AS month,
    EXTRACT(YEAR FROM d) AS year
FROM UNNEST(GENERATE_DATE_ARRAY('2024-01-01', '2026-12-31')) AS d
""")
print("dim_date created.")

# dim_campaign
run_bq(f"""
CREATE OR REPLACE TABLE `{G}.dim_campaign` AS
SELECT
    ROW_NUMBER() OVER (ORDER BY dnis) AS campaign_key,
    CAST(dnis AS STRING) AS dnis,
    campaign_name, client_name, channel
FROM `{R}.campaigns`
""")
print("dim_campaign created.")

# dim_disposition
run_bq(f"""
CREATE OR REPLACE TABLE `{G}.dim_disposition` AS
SELECT
    ROW_NUMBER() OVER (ORDER BY disposition) AS disposition_key,
    disposition AS disposition_name,
    disposition IN ('completed', 'sale') AS is_sale,
    disposition IN ('abandoned', 'hangup') AS is_abandon
FROM (SELECT DISTINCT LOWER(TRIM(disposition)) AS disposition FROM `{S}.calls`)
""")
print("dim_disposition created.")

print("\nAll dimensions created.")


In [ ]:
# === GOLD: Fact table ===
run_bq(f"""
CREATE OR REPLACE TABLE `{G}.fact_calls`
PARTITION BY call_date
CLUSTER BY campaign_key
AS
SELECT
    ROW_NUMBER() OVER (ORDER BY s.call_id) AS call_key,
    dd.date_key,
    s.call_hour_local AS hour_key,
    dc.campaign_key,
    di.disposition_key,
    s.call_id,
    s.call_date_local AS call_date,
    s.duration_sec,
    s.has_missing_duration,
    CASE WHEN o.order_id IS NOT NULL THEN TRUE ELSE FALSE END AS is_order,
    o.order_id,
    o.product_id,
    o.quantity,
    o.total_amount
FROM `{S}.calls` s
LEFT JOIN `{G}.dim_date` dd ON s.call_date_local = dd.full_date
LEFT JOIN `{G}.dim_campaign` dc ON CAST(s.dnis AS STRING) = dc.dnis
LEFT JOIN `{G}.dim_disposition` di ON s.disposition = di.disposition_name
LEFT JOIN `{S}.orders` o ON s.call_id = o.call_id
""")
print("fact_calls created.")


In [ ]:
# === QUALITY GATES ===
print("Gate 1: Duplicate check")
run_bq(f"""
SELECT call_id, COUNT(*) AS dupes
FROM `{G}.fact_calls` GROUP BY call_id HAVING COUNT(*) > 1
""")

print("Gate 2: Orphan keys")
run_bq(f"""
SELECT
    COUNTIF(campaign_key IS NULL) AS missing_campaign,
    COUNTIF(disposition_key IS NULL) AS missing_disposition,
    COUNT(*) AS total
FROM `{G}.fact_calls`
""")

print("Gate 3: Row count")
run_bq(f"""
SELECT 'silver.calls' AS layer, COUNT(*) AS rows FROM `{S}.calls`
UNION ALL
SELECT 'gold.fact_calls', COUNT(*) FROM `{G}.fact_calls`
""")

print("\nAll quality gates complete.")


#### Same Step in GCP Console: Gold Layer

1. In BigQuery Console, run each SQL above one at a time
2. Verify: expand `gold` dataset in left panel, you should see dim_date, dim_campaign, dim_disposition, fact_calls
3. Run the quality gate queries to validate
4. **You Should See:** Gate 1 returns 0 duplicates, Gate 2 shows few or no orphans, Gate 3 shows Silver and Gold counts close

### What Triggers This in Production?

In this notebook, you ran these steps manually. In production:

| Step | Triggered By |
|---|---|
| Bronze load | Cloud Function (automatic when file arrives) |
| Silver transform | Composer DAG schedules Dataproc job |
| Gold build | Composer DAG runs SQL after Silver completes |
| Quality gates | Composer DAG runs checks after Gold completes |
| Alert on failure | Composer sends email if any step fails |

The Composer DAG in Part 4 below shows exactly how this orchestration works.


**What just happened:**
1. You wrote a PySpark script (same dedup/clean logic as BigQuery SQL)
2. Uploaded it to GCS
3. Created a Dataproc cluster (2 workers)
4. Submitted the job (Spark distributed the work)
5. Silver Parquet files written to GCS
6. Loaded Silver Parquet into BigQuery (for Gold queries)
7. Deleted the cluster (stop paying)

**The difference from GCP_Full_Pipeline:** There, Silver was created by BigQuery SQL (`CREATE TABLE silver.calls AS SELECT ...`). Here, Silver was created by Spark writing Parquet to GCS, then loaded into BigQuery. Same result. Different engine. The Spark path scales to terabytes.

---

## Part 4: Airflow Orchestration - Tying the Pipeline Together

Everything above ran as individual steps: upload, transform, load, build Gold. In production, you need something to run those steps in order, retry on failure, and alert when things break. That something is **Apache Airflow**.

Airflow uses **DAGs** (Directed Acyclic Graphs) to define task dependencies. Each task is one step in your pipeline. Airflow handles scheduling, retries, logging, and alerting.

### Cloud Composer vs. Local Docker: Know the Cost

**Cloud Composer** is Google's managed Airflow service. It handles infrastructure, scaling, and upgrades. It is the production choice for GCP pipelines.

However, the smallest Cloud Composer environment costs **$300-500/month** just to keep running. That is not practical for learning. Do not create a Composer environment for practice.

**AWS equivalent:** Amazon MWAA (Managed Workflows for Apache Airflow) -- same concept, similar cost.

### Run Airflow Locally with Docker (Free)

For learning and development, run Airflow on your laptop using Docker. This gives you the real Airflow UI, real DAG execution, and real task logs -- identical to what Composer runs under the hood.

**docker-compose.yml** -- save this file in a folder called `airflow-local/`:

```yaml
# airflow-local/docker-compose.yml
# Lightweight Airflow for local development and learning
# Runs the same Airflow that Cloud Composer uses, on your laptop

version: '3.8'
services:
  airflow:
    image: apache/airflow:2.9.0
    environment:
      - AIRFLOW__CORE__EXECUTOR=SequentialExecutor
      - AIRFLOW__CORE__LOAD_EXAMPLES=False
      - AIRFLOW__DATABASE__SQL_ALCHEMY_CONN=sqlite:////opt/airflow/airflow.db
      - AIRFLOW__WEBSERVER__SECRET_KEY=local-dev-only
      - _AIRFLOW_DB_MIGRATE=true
      - _AIRFLOW_WWW_USER_CREATE=true
      - _AIRFLOW_WWW_USER_USERNAME=admin
      - _AIRFLOW_WWW_USER_PASSWORD=admin
    volumes:
      - ./dags:/opt/airflow/dags
      - ./logs:/opt/airflow/logs
    ports:
      - "8080:8080"
    command: airflow standalone
    healthcheck:
      test: ["CMD", "curl", "--fail", "http://localhost:8080/health"]
      interval: 30s
      timeout: 10s
      retries: 5
```

**Step-by-step setup:**

```bash
# 1. Create folder structure
mkdir -p airflow-local/dags airflow-local/logs

# 2. Save the docker-compose.yml above into airflow-local/

# 3. Start Airflow
cd airflow-local
docker compose up -d

# 4. Wait 30-60 seconds for initialization, then open browser
#    http://localhost:8080
#    Username: admin
#    Password: admin

# 5. When done, shut it down
docker compose down
```

Once Airflow is running locally, you deploy DAGs by copying Python files into the `dags/` folder. Airflow picks them up automatically within a few seconds.

In [ ]:
# === CLOUD COMPOSER: The DAG (Airflow workflow definition) ===
#
# COST WARNING: Cloud Composer costs $300-500/month minimum.
# For learning, run this DAG on local Docker Airflow (see cell above).
# For production, deploy to Cloud Composer.
#
# LOCAL DOCKER: Save as airflow-local/dags/pipeline_dag.py
#   cp pipeline_dag.py airflow-local/dags/
#   Airflow picks it up automatically within seconds.
#
# CLOUD COMPOSER: Deploy with gcloud
#   gcloud composer environments storage dags import \
#     --environment YOUR_COMPOSER_ENV \
#     --location us-central1 \
#     --source pipeline_dag.py

dag_code = """
# dags/call_center_pipeline.py
# Deploy this to Cloud Composer (or local Airflow via Docker)

from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

# -- Replace these with your values --
PROJECT_ID = "your-project-id"
REGION = "us-central1"
BUCKET = "your-bucket"
CLUSTER_NAME = "pipeline-silver-cluster"
DATASET_SILVER = "silver"
DATASET_GOLD = "gold"

default_args = {
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
    "email_on_failure": True,
    "email": ["oncall@company.com"],
}

dag = DAG(
    "call_center_pipeline",
    description="Full pipeline: Bronze -> Silver (Spark) -> Gold (BigQuery)",
    schedule_interval="0 2 * * *",   # 2 AM daily
    start_date=datetime(2026, 3, 1),
    catchup=False,
    default_args=default_args,
)

# ---- Task 1: Create Dataproc cluster ----
create_cluster = BashOperator(
    task_id="create_cluster",
    bash_command=(
        f"gcloud dataproc clusters create {CLUSTER_NAME} "
        f"--region={REGION} "
        f"--project={PROJECT_ID} "
        f"--num-workers=2 "
        f"--master-machine-type=n1-standard-2 "
        f"--worker-machine-type=n1-standard-2"
    ),
    dag=dag,
)

# ---- Task 2: Run Silver transform on Spark ----
silver_transform = BashOperator(
    task_id="silver_transform",
    bash_command=(
        f"gcloud dataproc jobs submit pyspark "
        f"gs://{BUCKET}/code/silver_transform.py "
        f"--cluster={CLUSTER_NAME} "
        f"--region={REGION} "
        f"--project={PROJECT_ID} "
        f"-- gs://{BUCKET}"
    ),
    dag=dag,
)

# ---- Task 3: Delete cluster (cost control) ----
delete_cluster = BashOperator(
    task_id="delete_cluster",
    bash_command=(
        f"gcloud dataproc clusters delete {CLUSTER_NAME} "
        f"--region={REGION} "
        f"--project={PROJECT_ID} "
        f"--quiet"
    ),
    trigger_rule="all_done",  # Delete even if transform failed
    dag=dag,
)

# ---- Task 4: Load Silver Parquet into BigQuery ----
load_silver_to_bq = BashOperator(
    task_id="load_silver_to_bq",
    bash_command=(
        f"bq load --source_format=PARQUET "
        f"--replace "
        f"{PROJECT_ID}:{DATASET_SILVER}.calls "
        f"gs://{BUCKET}/silver/calls/*.parquet"
    ),
    dag=dag,
)

# ---- Task 5: Build Gold marts ----
build_gold = BashOperator(
    task_id="build_gold_marts",
    bash_command=(
        f"bq query --use_legacy_sql=false "
        f"'CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_GOLD}.fact_calls` AS "
        f"SELECT * FROM `{PROJECT_ID}.{DATASET_SILVER}.calls`'"
    ),
    dag=dag,
)

# ---- Task 6: Quality gates ----
def run_quality_gates():
    from google.cloud import bigquery
    client = bigquery.Client()

    # Gate: no duplicates in fact table
    dupes = list(client.query(
        "SELECT call_id, COUNT(*) as cnt "
        "FROM gold.fact_calls GROUP BY call_id HAVING cnt > 1"
    ).result())

    if dupes:
        raise ValueError(f"QUALITY GATE FAILED: {len(dupes)} duplicate call_ids")
    print("All gates passed.")

quality_gates = PythonOperator(
    task_id="quality_gates",
    python_callable=run_quality_gates,
    dag=dag,
)

# ---- Dependencies (the DAG) ----
# This defines the execution order:
#
#   create_cluster -> silver_transform -> delete_cluster
#                                     -> load_silver_to_bq -> build_gold -> quality_gates

create_cluster >> silver_transform >> [delete_cluster, load_silver_to_bq]
load_silver_to_bq >> build_gold >> quality_gates
"""

print("=" * 60)
print("AIRFLOW DAG: call_center_pipeline")
print("=" * 60)
print("\nThis DAG runs at 2 AM daily and orchestrates:")
print("  1. Create Dataproc cluster")
print("  2. Run Silver transform (Spark)")
print("  3. Delete cluster (cost control)")
print("  4. Load Silver Parquet into BigQuery")
print("  5. Build Gold marts (BigQuery SQL)")
print("  6. Quality gates (fail = alert oncall)")
print("\n--- Execution flow ---")
print("create_cluster >> silver_transform >> delete_cluster")
print("                                  >> load_silver_to_bq >> build_gold >> quality_gates")
print("\nIf silver_transform fails:")
print("  - Retries 2 times (5 min apart)")
print("  - If still fails: cluster gets deleted (trigger_rule=all_done), email sent")
print("  - Gold does NOT run (stale data is worse than no data)")
print("\n--- How to run this DAG ---")
print("\nLOCAL DOCKER (free, for learning):")
print("  1. Save the DAG code above as: airflow-local/dags/pipeline_dag.py")
print("  2. Start Airflow: cd airflow-local && docker compose up -d")
print("  3. Open http://localhost:8080 (admin/admin)")
print("  4. Find 'call_center_pipeline' in the DAG list, enable it, trigger it")
print("\nCLOUD COMPOSER (production, $300-500/month):")
print("  1. Save as pipeline_dag.py")
print("  2. gcloud composer environments storage dags import \\")
print("       --environment YOUR_ENV --location us-central1 --source pipeline_dag.py")
print("  3. Open Composer > Airflow UI from GCP Console")

### Airflow UI Walkthrough: Local Docker

After running `docker compose up -d` and waiting 30-60 seconds:

1. **Open the UI:** Go to http://localhost:8080 in your browser
2. **Login:** Username `admin`, Password `admin`
3. **Find your DAG:** Look for `call_center_pipeline` in the DAG list on the home page. If it does not appear, check `airflow-local/dags/` for syntax errors in the Python file.
4. **Enable the DAG:** Toggle the switch on the left from "paused" to "active"
5. **Trigger a run:** Click the play button (triangle icon) on the right side of the DAG row. Select "Trigger DAG".
6. **Watch tasks execute:** Click the DAG name to open it. The **Graph** view shows tasks as boxes with colors:
   - Dark green = success
   - Red = failed
   - Light green = running
   - Yellow = upstream failed (skipped because a dependency failed)
7. **Check logs:** Click any task box, then click **Log** in the popup. This is where you debug failures. Every print statement and error traceback appears here.
8. **Task retries:** If a task fails, Airflow retries it based on `default_args` (2 retries, 5 minutes apart). You can see retry attempts in the log.

Note: On local Docker, the GCP commands (gcloud, bq) will fail unless you mount your credentials. For learning purposes, replace BashOperator commands with `echo` statements to trace the flow without actually calling GCP.

### Cloud Composer Console Walkthrough

If you are using Cloud Composer in production:

1. **Find Composer:** Go to [console.cloud.google.com/composer](https://console.cloud.google.com/composer)
2. **Open environment:** Click your Composer environment name
3. **Open Airflow UI:** Click the **Airflow webserver** link at the top of the environment details page. This opens the same Airflow UI you used locally -- same layout, same features.
4. **DAGs folder:** Composer stores DAGs in a GCS bucket. The environment details page shows the bucket path. You can also browse DAGs under the **DAGs** tab.
5. **Logs:** Available both in the Airflow UI (per task) and in Cloud Logging (search for `resource.type="cloud_composer_environment"`)
6. **Monitoring:** The Composer environment page shows CPU, memory, and DAG parse times. Use this to right-size your environment.

### Cost Reminder

Cloud Composer runs 24/7 even when no DAGs are scheduled. The minimum cost is approximately $300-500/month for the smallest environment. For learning and development, always use local Docker Airflow. Only deploy to Composer when you are ready for production workloads and have budget approval.

---

## Part 5: The Complete Automated Pipeline

### How Everything Connects in Production

**Incremental Load Flow:**
```
New raw data arrives --> Identify new rows (WHERE load_date > last_run) --> Transform only new rows --> MERGE into silver table
```
[See full diagram on GitHub](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/data/pipelines/cloud/04_Automation.md)

### The Two Rhythms of a Production Pipeline

| Rhythm | When | What | Component |
|---|---|---|---|
| **Event-driven** | Anytime a file arrives | Bronze load (append new data) | Cloud Function |
| **Scheduled** | 2 AM daily | Silver transform, Gold build, quality gates | Cloud Composer + Dataproc |

Bronze loads happen throughout the day as files arrive. The heavy transforms run once on a schedule.

### What You Built in This Notebook

| Component | What You Did | Production Role |
|---|---|---|
| **Pub/Sub** | Created topic, published, received message | Notification layer between services |
| **Cloud Functions** | Deployed function, triggered by file upload | Automated Bronze ingestion |
| **Dataproc** | Created cluster, submitted Spark job, deleted cluster | Scalable Silver transforms |
| **Cloud Composer** | Reviewed the DAG code | Orchestrates the entire flow |

### GCP to AWS Translation

| GCP | AWS | Role |
|---|---|---|
| Cloud Functions | Lambda | Event-driven triggers |
| Pub/Sub | SNS + SQS | Messaging |
| Dataproc | EMR | Managed Spark |
| Cloud Composer | MWAA | Managed Airflow |
| GCS | S3 | File storage |
| BigQuery | Redshift | Data warehouse |


In [ ]:
# === CLEANUP ===
# Uncomment to delete everything created in this notebook

# Delete Cloud Function
# !gcloud functions delete process_new_file --gen2 --region={REGION} --quiet

# Delete Pub/Sub
# !gcloud pubsub subscriptions delete {SUBSCRIPTION_NAME} --quiet
# !gcloud pubsub topics delete {TOPIC_NAME} --quiet

# Delete Dataproc cluster (if still running)
# !gcloud dataproc clusters delete {CLUSTER_NAME} --region={REGION} --quiet

# Delete test file
# !gcloud storage rm gs://{BUCKET_NAME}/bronze/calls/calls_test_trigger.json

print("Uncomment the lines above to clean up resources.")

#### Same Step in GCP Console: Cleanup

**Delete Cloud Function:**
1. Go to [console.cloud.google.com/functions](https://console.cloud.google.com/functions)
2. Check the box next to `process_new_file`
3. Click **"Delete"** at the top

**Delete Pub/Sub:**
1. Go to [console.cloud.google.com/cloudpubsub](https://console.cloud.google.com/cloudpubsub)
2. Click your topic > **"Delete topic"** (this also deletes subscriptions)

**Delete Dataproc cluster (if still running):**
1. Go to [console.cloud.google.com/dataproc](https://console.cloud.google.com/dataproc)
2. Check your cluster > **"Delete"**

**Delete test files:**
1. Go to Storage > your bucket > `bronze/` > delete the test file you uploaded

**Or keep everything** — the resources are small. Cloud Function and Pub/Sub cost nearly nothing when idle. Just make sure the Dataproc cluster is deleted (it costs ~$0.50/hour).
